# OpenBridge — 03 · Primera prueba real de GitHub

**Objetivo:** recuperar hasta cinco repositorios públicos mediante **GitHub REST API** y observar sus metadatos reales sin diseñar todavía tablas ni relaciones definitivas.

**Fuente exclusiva:** `https://api.github.com` · **Método:** GET · **Límite:** cinco solicitudes HTTP por ejecución autorizada · **Persistencia:** únicamente memoria y salidas visibles de la notebook · **PostgreSQL:** no se utiliza.

### Cómo interpretar el estado del experimento
- **Documentado:** propiedades indicadas por la documentación de GitHub, pendientes de contraste.
- **Observado:** valores obtenidos de respuestas HTTP de esta ejecución, con estado y fecha UTC.
- **Hipótesis:** posibles relaciones o reglas a comprobar con muestras posteriores.
- **Aprobado:** solamente las cinco consultas que figuran a continuación; **ninguna modificación del esquema SQL**.

> **Protección de ejecución:** la notebook se entrega con `ENABLE_LIVE_GITHUB = False`, sin resultados precargados. Para consultar GitHub desde tu computadora, sigue las instrucciones de la siguiente sección. No habilites la ejecución mientras no quieras consumir las cinco solicitudes.

## 1. Muestra y procedimiento autorizado

Repositorios seleccionados intencionalmente para observar variedad de lenguajes y metadatos; **no son una muestra estadísticamente representativa**:

| Repositorio | Endpoint exacto permitido |
|---|---|
| `apache/airflow` | `https://api.github.com/repos/apache/airflow` |
| `react/react` | `https://api.github.com/repos/react/react` |
| `torvalds/linux` | `https://api.github.com/repos/torvalds/linux` |
| `prometheus/prometheus` | `https://api.github.com/repos/prometheus/prometheus` |
| `BurntSushi/ripgrep` | `https://api.github.com/repos/BurntSushi/ripgrep` |

**Nota sobre identidad de React (investigación documental externa):** el repositorio consultado anteriormente como `facebook/react` se identifica ahora como `react/react`. GitHub conserva el ID externo `10270250` según la consulta pública revisada el 19-10-2026. Por eso la lista blanca usa la URL canónica actual y se conserva `id` junto con `full_name` en la observación. El ID indicado aquí es evidencia documental previa, **no un resultado de esta notebook**; debe contrastarse con el JSON obtenido en la ejecución real.

**Ejecución local:** abre el archivo en Jupyter o VS Code, selecciona el kernel Python del proyecto, ejecuta primero con el interruptor desactivado y comprueba que no haya llamadas. Cuando decidas efectuar la prueba, cambia **solo en tu sesión local** `ENABLE_LIVE_GITHUB = True` en la celda 2 y ejecuta las celdas en orden. No guardes el archivo con ese valor activado; vuelve a `False` y guarda la notebook con los resultados.

- No necesitas token para intentar consultar estos metadatos públicos. No se leen variables de entorno ni se imprimen credenciales.
- No se seguirán redirecciones, no habrá reintentos, búsquedas auxiliares ni paginación.
- Ejecutar nuevamente después de reiniciar el kernel podría emitir otras cinco solicitudes: **no repitas la extracción salvo nueva autorización**.
- Si una respuesta falla, se registra el error y se conservan las respuestas correctas. **Nunca se sustituyen por registros simulados.**

In [ ]:
import json
import math
from datetime import datetime, timezone
from time import monotonic
from collections import Counter

import pandas as pd
import requests
from IPython.display import display

# Cambiar a True solo durante una ejecución local deliberada; volver a False antes de guardar.
ENABLE_LIVE_GITHUB = False

APPROVED_ENDPOINTS = {
    "apache/airflow": "https://api.github.com/repos/apache/airflow",
    "react/react": "https://api.github.com/repos/react/react",
    "torvalds/linux": "https://api.github.com/repos/torvalds/linux",
    "prometheus/prometheus": "https://api.github.com/repos/prometheus/prometheus",
    "BurntSushi/ripgrep": "https://api.github.com/repos/BurntSushi/ripgrep",
}
assert len(APPROVED_ENDPOINTS) == 5 and len(set(APPROVED_ENDPOINTS.values())) == 5
assert all(url.startswith("https://api.github.com/repos/") for url in APPROVED_ENDPOINTS.values())

HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "OpenBridge-Exploracion/0.1",
}
print("Modo:", "CONSULTA REAL HABILITADA" if ENABLE_LIVE_GITHUB else "SEGURO: sin consultas")
print("Endpoints permitidos:", len(APPROVED_ENDPOINTS))

## 2. Función de extracción y bitácora

Cada consulta se realiza por separado con un timeout de 10 segundos y `allow_redirects=False`. La función devuelve **dos colecciones independientes**: respuestas JSON exitosas y registros de auditoría por intento. La marca de tiempo es UTC; `x-ratelimit-remaining` es informativa y no desencadena llamadas adicionales.

La función no se conecta a PostgreSQL, no descarga archivos ni usa otros servicios. La lista de URLs se define en el código y se valida antes de emitir solicitudes.

**Limitación:** los límites de GitHub pueden depender del origen de la llamada y de otras peticiones hechas desde la misma IP. Un 403 no implica siempre agotamiento de cuota.

In [ ]:
def _now_utc():
    return datetime.now(timezone.utc).isoformat()


def _http_error_kind(response):
    code = response.status_code
    if 300 <= code < 400:
        return "Redireccion_bloqueada"
    if code == 403:
        remaining = response.headers.get("x-ratelimit-remaining")
        message = ""
        try:
            body = response.json()
            if isinstance(body, dict):
                message = str(body.get("message", "")).lower()
        except (ValueError, TypeError):
            pass
        if remaining == "0" or "rate limit" in message:
            return "HTTP_403_limite_solicitudes"
        return "HTTP_403_acceso_denegado"
    if code == 404:
        return "HTTP_404_no_encontrado"
    return f"HTTP_{code}"


def fetch_five_github_repos(*, enabled=False, http_get=None):
    """Hasta 5 GET aprobados; sin reintentos, sin redirecciones ni fallbacks simulados."""
    if not enabled:
        return [], []

    getter = requests.get if http_get is None else http_get
    results, audit = [], []
    for slug, endpoint in APPROVED_ENDPOINTS.items():
        # Validación por igualdad exacta, no URLs construidas por datos externos.
        if endpoint != f"https://api.github.com/repos/{slug}":
            raise ValueError("Endpoint fuera de lista blanca")
        start = monotonic()
        entry = {
            "repository": slug,
            "endpoint": endpoint,
            "timestamp_utc": _now_utc(),
            "http_status": None,
            "success": False,
            "error_type": None,
            "rate_limit_remaining": None,
            "elapsed_seconds": None,
            "execution_mode": "LIVE_GITHUB",
            "is_simulated": False,
        }
        try:
            response = getter(endpoint, headers=HEADERS, timeout=10, allow_redirects=False)
            entry["http_status"] = response.status_code
            entry["rate_limit_remaining"] = response.headers.get("x-ratelimit-remaining")
            if response.status_code != 200:
                entry["error_type"] = _http_error_kind(response)
            else:
                try:
                    payload = response.json()
                except ValueError:
                    entry["error_type"] = "JSON_invalido"
                else:
                    if not isinstance(payload, dict):
                        entry["error_type"] = "JSON_no_objeto"
                    elif type(payload.get("id")) is not int or not isinstance(payload.get("full_name"), str):
                        entry["error_type"] = "Respuesta_sin_identificadores_validos"
                    elif payload["full_name"].casefold() != slug.casefold():
                        entry["error_type"] = "Identidad_repositorio_no_coincide"
                    else:
                        record = dict(payload)
                        record["_metadata_extracted_at_utc"] = entry["timestamp_utc"]
                        record["_metadata_endpoint_url"] = endpoint
                        record["_metadata_http_status"] = 200
                        record["_metadata_source"] = "GitHub REST API"
                        record["_metadata_execution_mode"] = "LIVE_GITHUB"
                        record["_metadata_is_simulated"] = False
                        results.append(record)
                        entry["success"] = True
        except requests.exceptions.Timeout:
            entry["error_type"] = "Timeout"
        except requests.exceptions.RequestException as exc:
            entry["error_type"] = type(exc).__name__
        finally:
            entry["elapsed_seconds"] = round(monotonic() - start, 3)
            audit.append(entry)
    return results, audit

print("Función preparada, sin solicitudes ejecutadas en esta celda.")

## 3. Ejecución real — celda controlada

**Esta es la única celda que puede iniciar peticiones externas.** Sin autorización local, su resultado es una colección vacía. Si la habilitas, realiza como máximo cinco GET, registra errores y no vuelve a ejecutar las consultas accidentalmente dentro de la misma sesión de kernel. Reiniciar el kernel reinicia esa protección.

Para conservar los resultados de la prueba, guarda la notebook después de analizarla; sus salidas podrán contener **metadatos públicos reales**. No se imprimen tokens ni cabeceras de autenticación.

In [ ]:
if ENABLE_LIVE_GITHUB:
    if globals().get("_OPENBRIDGE_LIVE_ALREADY_RUN", False):
        print("BLOQUEADO: la prueba real ya se intentó en este kernel. No se repiten solicitudes.")
    else:
        _OPENBRIDGE_LIVE_ALREADY_RUN = True  # impedir doble ejecución accidental
        live_payload, live_audit = fetch_five_github_repos(enabled=True)
        print(f"Consultas intentadas: {len(live_audit)}/5; exitosas: {len(live_payload)}")
else:
    live_payload, live_audit = [], []
    print("Modo seguro: 0 consultas HTTP. Cambia el interruptor solamente cuando autorices el experimento.")

## 4. Carga exploratoria en memoria y auditoría

**LOAD no significa insertar datos en PostgreSQL en este laboratorio.** Se construyen `df_repos` y `df_requests_log` a partir de los resultados de la celda anterior. Si todas las solicitudes fallan, las tablas quedan vacías y se informa de ello sin inventar datos.

Las columnas anidadas de GitHub se aplanan con `pd.json_normalize()`; `topics` permanece como lista para estudiarla por separado.

In [ ]:
AUDIT_COLUMNS = [
    "repository", "endpoint", "timestamp_utc", "http_status", "success", "error_type",
    "rate_limit_remaining", "elapsed_seconds", "execution_mode", "is_simulated"
]
df_requests_log = pd.DataFrame(live_audit, columns=AUDIT_COLUMNS)
df_repos = pd.json_normalize(live_payload, sep=".") if live_payload else pd.DataFrame()
print(f"Repositorios observados: {len(df_repos)} | Consultas registradas: {len(df_requests_log)}")
if not df_requests_log.empty:
    display(df_requests_log[["repository", "http_status", "success", "error_type", "rate_limit_remaining", "timestamp_utc"]])
if not df_repos.empty:
    visible = [c for c in ["id", "full_name", "owner.id", "language", "license.spdx_id", "_metadata_extracted_at_utc"] if c in df_repos.columns]
    display(df_repos[visible])
else:
    print("Sin datos reales disponibles para el análisis. No se usan fixtures como sustitutos.")

## 5. Calidad contractual sobre el JSON original

Las comprobaciones diferencian **campo ausente**, **nulo explícito** y **tipo inesperado**, incluso dentro de objetos anidados como `owner.id`. Para identificadores enteros se rechazan booleanos (`True` no debe tratarse como ID válido).

Estas reglas son **hipótesis de contrato de laboratorio**, no un esquema SQL aprobado. GitHub puede devolver campos opcionales vacíos; la evidencia de cinco respuestas no demuestra obligatoriedad global.

In [ ]:
FIELD_SPEC = {
    "id": (int, True, False),
    "node_id": (str, False, True),
    "full_name": (str, True, False),
    "name": (str, True, False),
    "html_url": (str, True, False),
    "owner.id": (int, True, False),
    "owner.login": (str, True, False),
    "language": (str, False, True),
    "description": (str, False, True),
    "license.spdx_id": (str, False, True),
    "topics": (list, False, True),
    "stargazers_count": (int, False, False),
    "forks_count": (int, False, False),
    "created_at": (str, False, True),
    "updated_at": (str, False, True),
}


def raw_path(record, path):
    value = record
    for part in path.split("."):
        if not isinstance(value, dict) or part not in value:
            return "AUSENTE", None
        value = value[part]
    return ("NULO_EXPLICITO", None) if value is None else ("PRESENTE", value)


def validate_payload(records):
    issues = []
    seen_ids = set()
    for n, record in enumerate(records):
        rid = record.get("id")
        if type(rid) is int and rid in seen_ids:
            issues.append({"fila": n, "campo": "id", "problema": "DUPLICADO"})
        if type(rid) is int:
            seen_ids.add(rid)
        for field, (type_expected, required, nullable) in FIELD_SPEC.items():
            status, value = raw_path(record, field)
            problem = None
            if status == "AUSENTE" and required:
                problem = "OBLIGATORIO_AUSENTE"
            elif status == "NULO_EXPLICITO" and not nullable:
                problem = "NULO_NO_PERMITIDO"
            elif status == "PRESENTE" and type(value) is not type_expected:
                problem = f"TIPO_INCORRECTO_{type(value).__name__}"
            if problem:
                issues.append({"fila": n, "campo": field, "problema": problem})
    return pd.DataFrame(issues, columns=["fila", "campo", "problema"])


df_contract_validation = validate_payload(live_payload)
print(f"Observaciones contractuales: {len(df_contract_validation)}")
if not df_contract_validation.empty:
    display(df_contract_validation)
else:
    print("Sin anomalías detectadas en los registros recuperados (o sin datos para validar).")

## 6. Análisis exploratorio con Pandas

Se exploran dimensiones, ejemplos, información de tipos, nulos, cardinalidad, duplicados y descriptivos. Las listas (`topics`) y objetos no hashables **no se incluyen** en `nunique()` global ni en `duplicated()` global: los duplicados se comprueban mediante `id` y `full_name`.

Con solo cinco repositorios las correlaciones entre métricas son ilustrativas; **no deben utilizarse para inferencia estadística ni causalidad**.

In [ ]:
if df_repos.empty:
    print("EDA pendiente: todavía no hay respuestas reales de GitHub.")
else:
    print("Forma (filas, columnas):", df_repos.shape)
    print("\nPrimeras filas: ")
    display(df_repos.head(3))
    print("\nÚltimas filas: ")
    display(df_repos.tail(2))
    print("\ninfo():")
    df_repos.info()
    print("\nTipos: ")
    display(df_repos.dtypes.rename("dtype").to_frame())
    print("\nResumen descriptivo (solo columnas representativas):")
    desc_cols = [c for c in ["stargazers_count", "forks_count", "open_issues_count", "language", "owner.type"] if c in df_repos]
    if desc_cols:
        display(df_repos[desc_cols].describe(include="all"))
    print("\nNulos y ausencias en Pandas (no distingue ausencia del JSON original):")
    display(df_repos.isna().sum().sort_values(ascending=False).head(30).rename("nulos").to_frame())
    scalar_cols = [c for c in df_repos if df_repos[c].map(lambda v: not isinstance(v, (list, dict, set))).all()]
    print("\nCardinalidad de columnas escalares:")
    display(df_repos[scalar_cols].nunique(dropna=False).sort_values(ascending=False).head(30).rename("unicos").to_frame())
    for key in ("id", "full_name"):
        if key in df_repos:
            print("Duplicados en", key, ":", int(df_repos.duplicated(subset=[key]).sum()))

In [ ]:
if df_repos.empty:
    print("Frecuencias y topics pendientes: no hay datos reales.")
else:
    for column in ("language", "owner.type", "license.spdx_id"):
        if column in df_repos:
            print(f"\nFrecuencias de {column} (nulos incluidos):")
            display(df_repos[column].value_counts(dropna=False))
    if "topics" in df_repos and "id" in df_repos:
        df_topics = df_repos[["id", "topics"]].explode("topics")
        topic_values = df_topics["topics"].dropna()
        print("\nAsociaciones de repositorio-topic observadas:", len(topic_values))
        display(topic_values.value_counts().head(15))
    else:
        df_topics = pd.DataFrame(columns=["id", "topics"])
    metrics = [c for c in ("stargazers_count", "forks_count", "open_issues_count", "subscribers_count") if c in df_repos]
    numeric = df_repos[metrics].apply(pd.to_numeric, errors="coerce") if metrics else pd.DataFrame()
    if len(numeric) >= 3 and len(metrics) >= 2:
        print("\nCorrelación Pearson — descriptiva; n muy pequeño:")
        display(numeric.corr(method="pearson"))
        print("\nCorrelación Spearman — descriptiva; n muy pequeño:")
        display(numeric.corr(method="spearman"))
    else:
        print("Correlaciones no calculadas por muestra o métricas insuficientes.")

## 7. Diccionario empírico inicial de campos

Los contadores se calculan sobre el **JSON original**, para conservar la diferencia entre:

- **Ausente:** la ruta no se encuentra en la respuesta.
- **Nulo explícito:** la ruta existe y contiene `null`.
- **Valor presente:** la ruta existe y tiene un valor no nulo.

El diccionario cubre campos prioritarios de GitHub. Es una primera muestra y no prueba que el contrato sea universal. Las rutas anidadas se comprueban completas (`owner.id`, no solo `owner`).

In [ ]:
def profile_fields(records, fields):
    rows = []
    for field in fields:
        counts = Counter()
        types = Counter()
        for record in records:
            status, value = raw_path(record, field)
            counts[status] += 1
            if status == "PRESENTE":
                types[type(value).__name__] += 1
        rows.append({
            "campo": field,
            "presentes": counts["PRESENTE"],
            "ausentes_json": counts["AUSENTE"],
            "nulos_explicitos_json": counts["NULO_EXPLICITO"],
            "tipos_observados": ", ".join(f"{t} ({n})" for t, n in sorted(types.items())) or "sin valores",
            "muestra_n": len(records),
        })
    return pd.DataFrame(rows)


df_diccionario = profile_fields(live_payload, FIELD_SPEC)
if live_payload:
    print("Diccionario preliminar, basado en respuestas reales:")
    display(df_diccionario)
else:
    print("Diccionario preparado, pendiente de observaciones reales.")
    display(df_diccionario.head(5))

## 8. Identificadores y relaciones candidatas

**No crear claves foráneas todavía.** El JSON de GitHub permite estudiar candidatos:

- `owner.id` → `id`: relación declarada entre propietario y repositorio, según el objeto anidado `owner` en cada respuesta. La cardinalidad y los cambios a lo largo del tiempo requieren investigación adicional.
- `id` (estable en GitHub) y `full_name` (puede cambiar al trasladar o renombrar repositorios): **no confundir identidad con ruta actual**. El caso `facebook/react` → `react/react` es un ejemplo documental pendiente de contrastar en esta ejecución.
- `id` → `topics[]`: asociación multivaluada; candidata a tabla de relaciones, sin esquema aprobado.
- `id` → `license.spdx_id`: atributo/identificador de licencia declarado en GitHub, que puede estar ausente o ser no concluyente. No se consulta SPDX.
- `stars`, `forks` y otras métricas son atributos cuantitativos, **no claves foráneas**.

Guardamos la evidencia y los identificadores originales antes de modelar. Un campo semejante o un nombre coincidente entre fuentes no demuestra equivalencia.

In [ ]:
if not df_repos.empty:
    relation_cols = [c for c in ("id", "full_name", "owner.id", "owner.login", "license.spdx_id", "topics") if c in df_repos]
    print("Candidatos relacionales — solo evidencia del payload de GitHub:")
    display(df_repos[relation_cols])
    print("Unicidad id:", df_repos["id"].is_unique if "id" in df_repos else "sin columna")
    if "owner.id" in df_repos:
        print("Cantidad de propietarios identificados en esta muestra:", df_repos["owner.id"].nunique(dropna=True))
else:
    print("Relaciones pendientes de observación: no se han recuperado repositorios reales.")

## 9. Informe de ejecución y próximos pasos

Completa estas conclusiones **después** de ejecutar el experimento real y revisar las salidas:

1. Fecha UTC de extracción y cantidad de GET efectivamente intentados.
2. Solicitudes exitosas, errores y cuota restante observada.
3. Número de registros obtenidos, anomalías del contrato y campos variables.
4. Evidencia de identificadores y relaciones candidatas; supuestos no demostrados.
5. Límites de la muestra de cinco repositorios y preguntas para la siguiente investigación.

**No se autoriza** ampliar el tamaño de muestra, ejecutar otras APIs, persistir en PostgreSQL ni establecer claves foráneas definitivas durante esta prueba. La notebook debe quedar con `ENABLE_LIVE_GITHUB = False` al guardarse, aunque conserve resultados reales en sus salidas.

In [ ]:
successful = int(df_requests_log["success"].sum()) if not df_requests_log.empty else 0
print("=== RESUMEN DE LA SESIÓN ===")
print("Modo configurado:", "LIVE_GITHUB" if ENABLE_LIVE_GITHUB else "SIN_RED")
print("GET registrados:", len(df_requests_log))
print("Solicitudes exitosas:", successful)
print("Repositorios válidos:", len(df_repos))
print("Observaciones contractuales:", len(df_contract_validation))
print("No se ha conectado PostgreSQL ni escrito archivos de datos desde esta notebook.")
if ENABLE_LIVE_GITHUB:
    print("IMPORTANTE: vuelve a poner ENABLE_LIVE_GITHUB=False antes de guardar el archivo.")

## 10. Autoprueba local de la función de extracción (sin red)

Esta celda usa un sustituto de `requests.get` que **nunca realiza tráfico HTTP**. Sirve para comprobar que un resultado 200 válido se conserva y que los otros cuatro escenarios (403, 404, timeout, JSON inválido) quedan registrados. Los objetos simulados **no se incorporan a `df_repos` ni al diccionario empírico**.

In [ ]:
def test_extractor_offline():
    assert "react/react" in APPROVED_ENDPOINTS
    assert "facebook/react" not in APPROVED_ENDPOINTS
    assert APPROVED_ENDPOINTS["react/react"] == "https://api.github.com/repos/react/react"
    class FakeResponse:
        def __init__(self, status, payload=None, headers=None, json_error=False):
            self.status_code = status
            self._payload = payload
            self.headers = headers or {}
            self._json_error = json_error
        def json(self):
            if self._json_error:
                raise ValueError("JSON inválido simulado")
            return self._payload

    calls = []
    def fake_get(url, **kwargs):
        calls.append(url)
        assert kwargs["allow_redirects"] is False
        assert kwargs["timeout"] == 10
        slug = url.split("/repos/", 1)[1]
        k = len(calls)
        if k == 1:
            return FakeResponse(200, {"id": 42, "full_name": slug, "owner": {"id": 7}})
        if k == 2:
            return FakeResponse(403, {"message": "API rate limit exceeded"}, {"x-ratelimit-remaining": "0"})
        if k == 3:
            return FakeResponse(404, {"message": "Not Found"})
        if k == 4:
            raise requests.exceptions.Timeout("Timeout simulado")
        return FakeResponse(200, json_error=True)

    records, log = fetch_five_github_repos(enabled=True, http_get=fake_get)
    assert len(calls) == 5 and len(log) == 5 and len(records) == 1
    assert [item["error_type"] for item in log] == [
        None, "HTTP_403_limite_solicitudes", "HTTP_404_no_encontrado", "Timeout", "JSON_invalido"
    ]
    assert not any(item["is_simulated"] for item in log)  # el modo del extractor es LIVE; el transporte fue simulado
    assert fetch_five_github_repos(enabled=False, http_get=fake_get) == ([], [])
    assert len(calls) == 5
    return "Autoprueba correcta: un resultado válido y cuatro errores aislados; 0 GET reales."

print(test_extractor_offline())